# Tema 5 - Respuesta en Frecuencia de Amplificadores

**Electronica General - 2o GIERM**

---

## Objetivos de aprendizaje

- Entender el concepto de polos y ceros en la funcion de transferencia de un amplificador
- Construir e interpretar diagramas de Bode de magnitud y fase
- Calcular el ancho de banda y el producto ganancia-ancho de banda (GBW)
- Aplicar el metodo de constantes de tiempo (Miller) para estimar el polo dominante
- Identificar las capacidades parasitas del transistor (MOS y BJT) y su efecto en la respuesta
- Comparar la respuesta de sistemas de un polo frente a dos polos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA = '#cb181d'       # rojo - limites, asintotas
COLOR_PUNTO = '#238b45'       # verde - puntos notables
COLOR_FASE = '#6a3d9a'        # morado - curvas de fase
COLOR_MILLER = '#e6550d'      # naranja - efecto Miller
COLOR_CLARO = '#a6cee3'       # azul claro - zonas de fondo

print('Configuracion lista.')

---

## 1. Introduccion: por que importa la frecuencia

Cuando analizamos un amplificador en DC o a frecuencias medias, usamos el **modelo de pequena senal** con capacidades en cortocircuito. Sin embargo, a **frecuencias altas**, las capacidades parasitas internas del transistor entran en juego y reducen la ganancia.

La respuesta en frecuencia nos dice:
- **Hasta que frecuencia** el amplificador mantiene su ganancia nominal
- **Como cae** la ganancia por encima (y por debajo, si hay acoplo AC) de esa banda
- **Que fase** introduce el amplificador en funcion de la frecuencia

El concepto clave es la **funcion de transferencia** $H(s)$, que relaciona la senal de salida con la de entrada en el dominio de Laplace.

---

## 2. Formulario completo

### 2.1 Funcion de transferencia general

$$\boxed{H(s) = A_m \cdot \frac{\prod_i (1 + s/z_i)}{\prod_j (1 + s/p_j)}}$$

donde $A_m$ es la ganancia en banda media, $z_i$ son los ceros y $p_j$ los polos.

### 2.2 Diagrama de Bode - Magnitud

$$\boxed{|H(j\omega)|_{\text{dB}} = 20 \log_{10} |H(j\omega)|}$$

### 2.3 Sistema de un polo

$$\boxed{H(s) = \frac{A_0}{1 + s/\omega_p}} \qquad |H(j\omega)| = \frac{A_0}{\sqrt{1 + (\omega/\omega_p)^2}}$$

- **Ancho de banda:** $BW = \omega_p = 2\pi f_p$
- A $\omega = \omega_p$: ganancia cae $3$ dB $\to$ $|H| = A_0/\sqrt{2}$
- Pendiente asintotica: $-20$ dB/decada por polo

### 2.4 Producto ganancia-ancho de banda (GBW)

$$\boxed{GBW = A_0 \cdot \omega_p = A_{CL} \cdot \omega_{CL}}$$

Es **constante** para sistemas de un polo: si aumentas la ganancia, pierdes ancho de banda y viceversa.

### 2.5 Efecto Miller

$$\boxed{C_{in} = C_{gs} + C_{gd}(1 + g_m R_L)}$$

La capacidad $C_{gd}$ (o $C_{\mu}$ en BJT) se **multiplica** por la ganancia del amplificador, creando el **polo dominante** en la entrada.

### 2.6 Metodo de constantes de tiempo

$$\boxed{\omega_p \approx \frac{1}{\sum_i R_i C_i}}$$

donde $R_i$ es la resistencia vista por cada capacidad $C_i$ cuando las demas estan en circuito abierto (altas frecuencias).

### 2.7 Capacidades parasitas del transistor

| Transistor | Capacidades | Origen fisico |
|------------|-------------|---------------|
| MOSFET | $C_{gs}$, $C_{gd}$, $C_{ds}$ | Overlap gate-canal, capacidad de juntura |
| BJT | $C_{\pi}$, $C_{\mu}$, $C_{cs}$ | Difusion B-E, juntura B-C, colector-sustrato |

Relacion de equivalencia: $C_{\pi} \leftrightarrow C_{gs}$, $C_{\mu} \leftrightarrow C_{gd}$

---

## 3. Diagrama de Bode: sistema de un polo

Vamos a representar la respuesta en frecuencia de un amplificador con funcion de transferencia:

$$H(s) = \frac{A_0}{1 + s/\omega_p}$$

con $A_0 = 100$ (40 dB) y $f_p = 10$ kHz.

In [ ]:
# Diagrama de Bode - Sistema de un polo
A0 = 100       # ganancia en banda media
fp = 10e3      # polo en Hz
wp = 2 * np.pi * fp

# Funcion de transferencia: H(s) = A0 / (1 + s/wp)
num = [A0]
den = [1/wp, 1]
sys = signal.TransferFunction(num, den)

# Rango de frecuencias
w = np.logspace(1, 8, 1000)  # rad/s
w_eval, mag, phase = signal.bode(sys, w)
f_eval = w_eval / (2 * np.pi)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Magnitud
ax1.semilogx(f_eval, mag, color=COLOR_PRINCIPAL, lw=2.5, label=r'$|H(j\omega)|$')
ax1.axhline(y=20*np.log10(A0), color=COLOR_RECTA, ls='--', lw=1.5, alpha=0.7, label=f'$A_0$ = {20*np.log10(A0):.0f} dB')
ax1.axhline(y=20*np.log10(A0)-3, color=COLOR_PUNTO, ls='--', lw=1.5, alpha=0.7, label=r'$A_0 - 3$ dB')
ax1.axvline(x=fp, color='gray', ls=':', lw=1, alpha=0.6)
ax1.plot(fp, 20*np.log10(A0)-3, 'o', color=COLOR_PUNTO, ms=10, zorder=5)
ax1.annotate(f'$f_p = {fp/1e3:.0f}$ kHz\n$-3$ dB', xy=(fp, 20*np.log10(A0)-3),
             xytext=(fp*5, 20*np.log10(A0)-1), fontsize=11, color=COLOR_PUNTO,
             arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=1.5),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PUNTO))
ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title('Diagrama de Bode - Magnitud', fontweight='bold')
ax1.legend(fontsize=11, loc='lower left')
ax1.grid(True, alpha=0.3, which='both')
ax1.set_xlim(1, 1e7)

# Fase
ax2.semilogx(f_eval, phase, color=COLOR_FASE, lw=2.5, label=r'$\angle H(j\omega)$')
ax2.axhline(y=-45, color='gray', ls=':', lw=1, alpha=0.6)
ax2.axvline(x=fp, color='gray', ls=':', lw=1, alpha=0.6)
ax2.plot(fp, -45, 'o', color=COLOR_FASE, ms=10, zorder=5)
ax2.annotate(r'$-45^{\circ}$ en $f_p$', xy=(fp, -45),
             xytext=(fp*5, -30), fontsize=11, color=COLOR_FASE,
             arrowprops=dict(arrowstyle='->', color=COLOR_FASE, lw=1.5),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_FASE))
ax2.set_xlabel('Frecuencia (Hz)')
ax2.set_ylabel(r'Fase ($^{\circ}$)')
ax2.set_title('Diagrama de Bode - Fase', fontweight='bold')
ax2.legend(fontsize=11, loc='lower left')
ax2.grid(True, alpha=0.3, which='both')
ax2.set_xlim(1, 1e7)
ax2.set_ylim(-100, 5)

plt.tight_layout()
plt.show()

---

## 4. Diagrama de polos y ceros

Representamos los polos y ceros en el plano $s$ (plano complejo). Un sistema con dos polos reales y un cero:

In [ ]:
# Diagrama de polos y ceros
# Sistema con 2 polos y 1 cero
zeros = [-2*np.pi*50e3]  # cero en -50 kHz
poles = [-2*np.pi*10e3, -2*np.pi*500e3]  # polos en -10 kHz y -500 kHz

fig, ax = plt.subplots(figsize=(10, 7))

# Ejes del plano s
ax.axhline(y=0, color='gray', lw=0.8)
ax.axvline(x=0, color='gray', lw=0.8)

# Polos (X roja)
for p in poles:
    ax.plot(p/(2*np.pi*1e3), 0, 'x', color=COLOR_RECTA, ms=14, mew=3, zorder=5)
ax.plot([], [], 'x', color=COLOR_RECTA, ms=12, mew=3, label='Polos')

# Ceros (O azul)
for z in zeros:
    ax.plot(z/(2*np.pi*1e3), 0, 'o', color=COLOR_PRINCIPAL, ms=12, mew=2,
            markerfacecolor='white', markeredgecolor=COLOR_PRINCIPAL, zorder=5)
ax.plot([], [], 'o', color=COLOR_PRINCIPAL, ms=10, mew=2,
        markerfacecolor='white', markeredgecolor=COLOR_PRINCIPAL, label='Ceros')

# Anotaciones
ax.annotate(r'$p_1 = -2\pi \cdot 10$ kHz', xy=(poles[0]/(2*np.pi*1e3), 0),
            xytext=(-8, 2.5), fontsize=12, color=COLOR_RECTA,
            arrowprops=dict(arrowstyle='->', color=COLOR_RECTA, lw=1.5))
ax.annotate(r'$p_2 = -2\pi \cdot 500$ kHz', xy=(poles[1]/(2*np.pi*1e3), 0),
            xytext=(-450, 2.5), fontsize=12, color=COLOR_RECTA,
            arrowprops=dict(arrowstyle='->', color=COLOR_RECTA, lw=1.5))
ax.annotate(r'$z_1 = -2\pi \cdot 50$ kHz', xy=(zeros[0]/(2*np.pi*1e3), 0),
            xytext=(-45, -3), fontsize=12, color=COLOR_PRINCIPAL,
            arrowprops=dict(arrowstyle='->', color=COLOR_PRINCIPAL, lw=1.5))

# Zona estable (semiplano izquierdo)
ax.axvspan(ax.get_xlim()[0], 0, alpha=0.08, color=COLOR_CLARO, label='Semiplano estable')

ax.set_xlabel(r'$\sigma / (2\pi)$ (kHz)', fontsize=13)
ax.set_ylabel(r'$j\omega / (2\pi)$ (kHz)', fontsize=13)
ax.set_title('Diagrama de Polos y Ceros en el plano s', fontweight='bold')
ax.legend(fontsize=12, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 5. Reglas para construir diagramas de Bode

### 5.1 Reglas de magnitud

| Elemento | Efecto en magnitud | Pendiente |
|----------|-------------------|----------|
| Polo en $\omega_p$ | Cae $-3$ dB en $\omega_p$, luego $-20$ dB/dec | $-20$ dB/decada |
| Cero en $\omega_z$ | Sube $+3$ dB en $\omega_z$, luego $+20$ dB/dec | $+20$ dB/decada |
| Doble polo en $\omega_p$ | Cae $-6$ dB en $\omega_p$, luego $-40$ dB/dec | $-40$ dB/decada |
| Ganancia $A_0$ | Desplazamiento vertical constante | $0$ dB/decada |

### 5.2 Reglas de fase

| Elemento | Fase en $\omega = 0$ | Fase en $\omega_p$ | Fase en $\omega \to \infty$ |
|----------|---------------------|--------------------|--------------------------|
| Polo | $0^\circ$ | $-45^\circ$ | $-90^\circ$ |
| Cero | $0^\circ$ | $+45^\circ$ | $+90^\circ$ |

**Regla practica:** La transicion de fase ocurre entre $0.1\omega_p$ y $10\omega_p$ (dos decadas centradas en el polo/cero).

In [ ]:
# Diagrama de Bode: asintotas vs respuesta real
A0_2p = 1000  # 60 dB
fp1 = 1e4     # primer polo
fp2 = 1e6     # segundo polo

wp1 = 2 * np.pi * fp1
wp2 = 2 * np.pi * fp2

# H(s) = A0 / ((1+s/wp1)*(1+s/wp2))
num_2p = [A0_2p]
den_2p = np.polymul([1/wp1, 1], [1/wp2, 1])
sys_2p = signal.TransferFunction(num_2p, den_2p)

w_2p = np.logspace(2, 9, 2000)
w_2p_eval, mag_2p, phase_2p = signal.bode(sys_2p, w_2p)
f_2p = w_2p_eval / (2 * np.pi)

# Asintotas de magnitud
mag_asymp = np.zeros_like(f_2p)
for i, f in enumerate(f_2p):
    val = 20 * np.log10(A0_2p)
    if f > fp1:
        val -= 20 * np.log10(f / fp1)
    if f > fp2:
        val -= 20 * np.log10(f / fp2)
    mag_asymp[i] = val

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Magnitud
ax1.semilogx(f_2p, mag_2p, color=COLOR_PRINCIPAL, lw=2.5, label='Respuesta real')
ax1.semilogx(f_2p, mag_asymp, color=COLOR_RECTA, lw=2, ls='--', label='Asintotas')
ax1.axvline(x=fp1, color='gray', ls=':', lw=1, alpha=0.5)
ax1.axvline(x=fp2, color='gray', ls=':', lw=1, alpha=0.5)
ax1.annotate(f'$f_{{p1}} = {fp1/1e3:.0f}$ kHz', xy=(fp1, 57), fontsize=11, color='gray')
ax1.annotate(f'$f_{{p2}} = {fp2/1e6:.0f}$ MHz', xy=(fp2, 37), fontsize=11, color='gray')
ax1.text(3e2, 52, r'$-20$ dB/dec', fontsize=11, color=COLOR_RECTA, rotation=-18)
ax1.text(3e6, 10, r'$-40$ dB/dec', fontsize=11, color=COLOR_RECTA, rotation=-35)
ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title('Bode con asintotas - Sistema de dos polos', fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, which='both')

# Fase
ax2.semilogx(f_2p, phase_2p, color=COLOR_FASE, lw=2.5, label=r'$\angle H(j\omega)$')
ax2.axhline(y=-90, color='gray', ls=':', lw=1, alpha=0.5)
ax2.axhline(y=-180, color='gray', ls=':', lw=1, alpha=0.5)
ax2.set_xlabel('Frecuencia (Hz)')
ax2.set_ylabel(r'Fase ($^{\circ}$)')
ax2.set_title('Fase - Sistema de dos polos', fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---

## 6. Efecto Miller en amplificador source comun (CS)

El amplificador CS es el caso clasico donde el efecto Miller domina la respuesta en frecuencia.

**Modelo de alta frecuencia del MOSFET:**
- $C_{gs}$: entre gate y source (tipicamente $\sim 1$ pF)
- $C_{gd}$: entre gate y drain (tipicamente $\sim 0.2$ pF) $\to$ esta se **multiplica por Miller**
- $C_{ds}$: entre drain y source (generalmente despreciable)

**Polo dominante por efecto Miller:**

$$\boxed{f_{p,in} = \frac{1}{2\pi \cdot R_{sig} \cdot [C_{gs} + C_{gd}(1 + g_m R_L)]}}$$

$$f_{p,out} = \frac{1}{2\pi \cdot R_L \cdot [C_{ds} + C_{gd}]}$$

Donde $R_{sig}$ es la resistencia de la fuente de senal.

**Nota clave:** $C_{gd}$ se multiplica por $(1 + g_m R_L)$, que puede ser $\times 50$ o mas. Una capacidad de $0.2$ pF se convierte en $10$ pF equivalente en la entrada.

In [ ]:
# Respuesta en frecuencia del amplificador CS con efecto Miller
# Parametros tipicos
gm = 5e-3       # transconductancia (5 mA/V)
RL = 10e3       # resistencia de carga (10 kOhm)
Rsig = 5e3      # resistencia de fuente (5 kOhm)
Cgs = 1e-12     # 1 pF
Cgd = 0.2e-12   # 0.2 pF
Cds = 0.1e-12   # 0.1 pF (despreciable)

# Ganancia en banda media
Av = -gm * RL  # = -50
A0_cs = abs(Av)

# Capacidad Miller en la entrada
C_Miller = Cgd * (1 + gm * RL)
Cin_total = Cgs + C_Miller

# Polos
fp_in = 1 / (2 * np.pi * Rsig * Cin_total)
fp_out = 1 / (2 * np.pi * RL * (Cds + Cgd))

print(f'Ganancia en banda media: Av = {Av:.0f} ({20*np.log10(A0_cs):.1f} dB)')
print(f'C_Miller = Cgd*(1 + gm*RL) = {C_Miller*1e12:.1f} pF')
print(f'Cin total = Cgs + C_Miller = {Cin_total*1e12:.1f} pF')
print(f'Polo de entrada (dominante): fp_in = {fp_in/1e6:.2f} MHz')
print(f'Polo de salida: fp_out = {fp_out/1e6:.1f} MHz')
print(f'Separacion de polos: fp_out/fp_in = {fp_out/fp_in:.1f}x')

In [ ]:
# Diagrama de Bode del amplificador CS con efecto Miller
wp_in = 2 * np.pi * fp_in
wp_out = 2 * np.pi * fp_out

# H(s) = A0 / ((1 + s/wp_in) * (1 + s/wp_out))
num_cs = [A0_cs]
den_cs = np.polymul([1/wp_in, 1], [1/wp_out, 1])
sys_cs = signal.TransferFunction(num_cs, den_cs)

w_cs = np.logspace(3, 10, 2000)
w_cs_eval, mag_cs, phase_cs = signal.bode(sys_cs, w_cs)
f_cs = w_cs_eval / (2 * np.pi)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Magnitud
ax1.semilogx(f_cs, mag_cs, color=COLOR_PRINCIPAL, lw=2.5, label=r'$|A_v(j\omega)|$')
ax1.axhline(y=20*np.log10(A0_cs), color=COLOR_RECTA, ls='--', lw=1.5, alpha=0.6)
ax1.axhline(y=20*np.log10(A0_cs)-3, color=COLOR_PUNTO, ls='--', lw=1.5, alpha=0.6, label=r'$-3$ dB')
ax1.axvline(x=fp_in, color=COLOR_MILLER, ls=':', lw=1.5, alpha=0.7)
ax1.axvline(x=fp_out, color='gray', ls=':', lw=1.5, alpha=0.7)
ax1.plot(fp_in, 20*np.log10(A0_cs)-3, 'o', color=COLOR_MILLER, ms=10, zorder=5)
ax1.annotate(f'$f_{{p,in}}$ = {fp_in/1e6:.2f} MHz\n(polo Miller)', xy=(fp_in, 20*np.log10(A0_cs)-3),
             xytext=(fp_in*0.05, 20*np.log10(A0_cs)+2), fontsize=11, color=COLOR_MILLER,
             arrowprops=dict(arrowstyle='->', color=COLOR_MILLER, lw=1.5),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_MILLER))
ax1.annotate(f'$f_{{p,out}}$ = {fp_out/1e6:.0f} MHz', xy=(fp_out, 20*np.log10(A0_cs)-20),
             xytext=(fp_out*3, 20*np.log10(A0_cs)-10), fontsize=11, color='gray',
             arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))
ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title('Respuesta en frecuencia - Amplificador CS con efecto Miller', fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, which='both')

# Fase
ax2.semilogx(f_cs, phase_cs, color=COLOR_FASE, lw=2.5)
ax2.axhline(y=-90, color='gray', ls=':', lw=1, alpha=0.5)
ax2.axhline(y=-180, color='gray', ls=':', lw=1, alpha=0.5)
ax2.set_xlabel('Frecuencia (Hz)')
ax2.set_ylabel(r'Fase ($^{\circ}$)')
ax2.set_title('Fase - Amplificador CS', fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---

## 7. Producto Ganancia-Ancho de Banda (GBW)

Para un amplificador de **un polo dominante**, el GBW es constante:

$$GBW = A_0 \cdot f_p = A_0 \cdot BW$$

Esto significa que:
- Si **realimentamos** el amplificador para reducir la ganancia a $A_{CL}$, el ancho de banda aumenta a $BW_{CL}$
- Se cumple: $A_{CL} \cdot BW_{CL} = A_0 \cdot f_p = \text{constante}$

**Ejemplo numerico:**
- $A_0 = 100$ (40 dB), $f_p = 10$ kHz $\to$ $GBW = 100 \times 10\text{k} = 1$ MHz
- Si $A_{CL} = 10$ (20 dB) $\to$ $BW_{CL} = 1\text{M}/10 = 100$ kHz
- Si $A_{CL} = 1$ (0 dB) $\to$ $BW_{CL} = 1$ MHz (unidad de ganancia)

In [ ]:
# Demostracion visual del producto ganancia-ancho de banda (GBW)
A0_gbw = 100
fp_gbw = 10e3
GBW = A0_gbw * fp_gbw  # 1 MHz

ganancias = [100, 50, 20, 10, 5, 2, 1]
colores_gbw = ['#08306b', '#08519c', '#2171b5', '#4292c6', '#6baed6', '#9ecae1', '#c6dbef']

fig, ax = plt.subplots(figsize=(12, 7))

for A_cl, col in zip(ganancias, colores_gbw):
    bw_cl = GBW / A_cl
    f_range = np.logspace(1, 7, 1000)
    mag_gbw = 20 * np.log10(A_cl / np.sqrt(1 + (f_range / bw_cl)**2))
    ax.semilogx(f_range, mag_gbw, color=col, lw=2,
                label=f'$A_{{CL}}={A_cl}$, BW={bw_cl/1e3:.0f} kHz')
    # Marcar punto -3 dB
    ax.plot(bw_cl, 20*np.log10(A_cl) - 3, 'o', color=col, ms=7, zorder=5)

# Linea de GBW
f_gbw_line = np.logspace(np.log10(fp_gbw), 7, 100)
mag_gbw_line = 20 * np.log10(GBW / f_gbw_line)
ax.semilogx(f_gbw_line, mag_gbw_line, color=COLOR_RECTA, lw=2.5, ls='--',
            label=f'GBW = {GBW/1e6:.0f} MHz')

ax.set_xlabel('Frecuencia (Hz)')
ax.set_ylabel(r'Ganancia (dB)')
ax.set_title('Producto Ganancia-Ancho de Banda (GBW) constante', fontweight='bold')
ax.legend(fontsize=10, loc='lower left', ncol=2)
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(10, 5e6)
ax.set_ylim(-10, 45)

plt.tight_layout()
plt.show()

print(f'GBW = A0 x fp = {A0_gbw} x {fp_gbw/1e3:.0f} kHz = {GBW/1e6:.0f} MHz')

---

## 8. Comparacion: un polo vs dos polos

La separacion entre polos determina el comportamiento:
- **Polo dominante bien separado:** se comporta casi como un sistema de un polo
- **Polos cercanos:** la caida es mas abrupta y la fase cambia mas rapido $\to$ peor para realimentacion

In [ ]:
# Comparacion: 1 polo vs 2 polos (separados vs cercanos)
A0_comp = 100  # 40 dB
fp1_comp = 10e3

# Caso 1: un solo polo
w_1p = 2 * np.pi * fp1_comp
sys_1p = signal.TransferFunction([A0_comp], [1/w_1p, 1])

# Caso 2: dos polos bien separados (factor 100)
fp2_sep = 1e6
w_2p_sep = 2 * np.pi * fp2_sep
den_2p_sep = np.polymul([1/w_1p, 1], [1/w_2p_sep, 1])
sys_2p_sep = signal.TransferFunction([A0_comp], den_2p_sep)

# Caso 3: dos polos cercanos (factor 5)
fp2_close = 50e3
w_2p_close = 2 * np.pi * fp2_close
den_2p_close = np.polymul([1/w_1p, 1], [1/w_2p_close, 1])
sys_2p_close = signal.TransferFunction([A0_comp], den_2p_close)

w_comp = np.logspace(2, 9, 2000)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9))

# Magnitud
for sys_i, label_i, col_i, lw_i in [
    (sys_1p, '1 polo', COLOR_PRINCIPAL, 2.5),
    (sys_2p_sep, '2 polos (sep. x100)', COLOR_PUNTO, 2),
    (sys_2p_close, '2 polos (sep. x5)', COLOR_RECTA, 2)]:
    w_e, m_e, _ = signal.bode(sys_i, w_comp)
    ax1.semilogx(w_e/(2*np.pi), m_e, color=col_i, lw=lw_i, label=label_i)

ax1.axhline(y=20*np.log10(A0_comp)-3, color='gray', ls='--', lw=1, alpha=0.5)
ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title('Comparacion de magnitud: 1 polo vs 2 polos', fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, which='both')

# Fase
for sys_i, label_i, col_i, lw_i in [
    (sys_1p, '1 polo', COLOR_PRINCIPAL, 2.5),
    (sys_2p_sep, '2 polos (sep. x100)', COLOR_PUNTO, 2),
    (sys_2p_close, '2 polos (sep. x5)', COLOR_RECTA, 2)]:
    w_e, _, p_e = signal.bode(sys_i, w_comp)
    ax2.semilogx(w_e/(2*np.pi), p_e, color=col_i, lw=lw_i, label=label_i)

ax2.axhline(y=-90, color='gray', ls=':', lw=1, alpha=0.5)
ax2.axhline(y=-180, color='gray', ls=':', lw=1, alpha=0.5)
ax2.set_xlabel('Frecuencia (Hz)')
ax2.set_ylabel(r'Fase ($^{\circ}$)')
ax2.set_title('Comparacion de fase: 1 polo vs 2 polos', fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---

## 9. Metodo de las constantes de tiempo

Este metodo permite estimar el polo dominante $\omega_H$ **sin resolver la funcion de transferencia completa**.

### Procedimiento:

1. **Identificar** todas las capacidades parasitas ($C_1, C_2, \ldots, C_n$)
2. Para **cada** capacidad $C_i$:
   - Poner las demas capacidades en **circuito abierto**
   - Anular fuentes independientes (tension $\to$ cortocircuito, corriente $\to$ abierto)
   - Calcular la **resistencia equivalente** $R_i$ vista por $C_i$
3. La frecuencia del polo dominante es:

$$\boxed{\omega_H \approx \frac{1}{\sum_{i=1}^{n} R_i C_i} = \frac{1}{\tau_1 + \tau_2 + \cdots + \tau_n}}$$

### Ejemplo para CS con efecto Miller:

- **$C_{gs}$:** ve $R_{sig}$ $\to$ $\tau_1 = R_{sig} \cdot C_{gs}$
- **$C_{gd}$:** ve $R_{sig}(1 + g_m R_L) + R_L$ $\to$ $\tau_2 = [R_{sig}(1 + g_m R_L) + R_L] \cdot C_{gd}$
- **$C_{ds}$:** ve $R_L$ $\to$ $\tau_3 = R_L \cdot C_{ds}$

$$\omega_H \approx \frac{1}{\tau_1 + \tau_2 + \tau_3}$$

**Nota:** Este metodo da una **aproximacion** del polo dominante. Es exacto cuando hay un polo claramente dominante.

In [ ]:
# Metodo de constantes de tiempo - Ejemplo numerico CS
# Parametros (los mismos de la seccion 6)
print('=== Metodo de constantes de tiempo para CS ===')
print(f'Parametros: gm = {gm*1e3:.0f} mA/V, RL = {RL/1e3:.0f} kOhm, Rsig = {Rsig/1e3:.0f} kOhm')
print(f'Capacidades: Cgs = {Cgs*1e12:.1f} pF, Cgd = {Cgd*1e12:.1f} pF, Cds = {Cds*1e12:.1f} pF')
print()

# Constante de tiempo de Cgs
tau1 = Rsig * Cgs
print(f'tau_1 = Rsig * Cgs = {Rsig/1e3:.0f}k x {Cgs*1e12:.1f}p = {tau1*1e9:.2f} ns')

# Constante de tiempo de Cgd (incluyendo efecto Miller)
R_Cgd = Rsig * (1 + gm * RL) + RL
tau2 = R_Cgd * Cgd
print(f'tau_2 = [Rsig*(1+gm*RL) + RL] * Cgd')
print(f'      = [{Rsig/1e3:.0f}k*(1+{gm*1e3:.0f}m*{RL/1e3:.0f}k) + {RL/1e3:.0f}k] * {Cgd*1e12:.1f}p')
print(f'      = [{R_Cgd/1e3:.0f} kOhm] * {Cgd*1e12:.1f} pF = {tau2*1e9:.2f} ns')

# Constante de tiempo de Cds
tau3 = RL * Cds
print(f'tau_3 = RL * Cds = {RL/1e3:.0f}k x {Cds*1e12:.1f}p = {tau3*1e9:.2f} ns')

# Polo dominante
tau_total = tau1 + tau2 + tau3
fH_approx = 1 / (2 * np.pi * tau_total)
print(f'\nSuma de taus = {tau_total*1e9:.2f} ns')
print(f'fH (aprox) = 1/(2*pi*tau_total) = {fH_approx/1e6:.2f} MHz')
print(f'fH (exacto, del modelo) = {fp_in/1e6:.2f} MHz')
print(f'Error del metodo: {abs(fH_approx-fp_in)/fp_in*100:.1f}%')

---

## 10. Capacidades parasitas: MOS vs BJT

### 10.1 MOSFET

| Capacidad | Ubicacion | Valor tipico | Efecto principal |
|-----------|-----------|-------------|------------------|
| $C_{gs}$ | Gate-Source | $1 - 5$ pF | Limita $f_T$ |
| $C_{gd}$ | Gate-Drain | $0.1 - 1$ pF | **Efecto Miller** (dominante) |
| $C_{ds}$ | Drain-Source | $0.05 - 0.5$ pF | Generalmente despreciable |

### 10.2 BJT

| Capacidad | Ubicacion | Valor tipico | Efecto principal |
|-----------|-----------|-------------|------------------|
| $C_{\pi}$ | Base-Emisor | $5 - 50$ pF | Limita $f_T$ |
| $C_{\mu}$ | Base-Colector | $0.5 - 5$ pF | **Efecto Miller** (dominante) |
| $C_{cs}$ | Colector-Sustrato | $1 - 10$ pF | Polo de salida |

### 10.3 Frecuencia de transicion $f_T$

$$\boxed{f_T = \frac{g_m}{2\pi (C_{\pi} + C_{\mu})}} \quad \text{(BJT)} \qquad \boxed{f_T = \frac{g_m}{2\pi (C_{gs} + C_{gd})}} \quad \text{(MOS)}$$

$f_T$ es la frecuencia a la que la ganancia de corriente cae a $1$ (0 dB).

In [ ]:
# Modelo de alta frecuencia: MOS vs BJT (diagrama esquematico)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- MOSFET ---
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 8)
ax1.set_aspect('equal')
ax1.axis('off')
ax1.set_title('Modelo AC alta frecuencia - MOSFET', fontsize=14, fontweight='bold')

# Terminales
for name, xy in [('G', (1, 4)), ('D', (8, 6.5)), ('S', (8, 1.5))]:
    ax1.plot(*xy, 'o', color=COLOR_PRINCIPAL, ms=10, zorder=5)
    ax1.text(xy[0]-0.4 if name=='G' else xy[0]+0.4, xy[1], name,
             fontsize=14, fontweight='bold', color=COLOR_PRINCIPAL, ha='center', va='center')

# gm*Vgs fuente de corriente
ax1.annotate('', xy=(8, 5.5), xytext=(8, 2.5),
             arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2))
ax1.text(9.2, 4, r'$g_m V_{gs}$', fontsize=13, color=COLOR_PUNTO, ha='center')

# Capacidades
# Cgs
ax1.annotate('', xy=(3.5, 2), xytext=(1.5, 3.5),
             arrowprops=dict(arrowstyle='-', color=COLOR_MILLER, lw=2))
ax1.text(2, 2.2, r'$C_{gs}$', fontsize=13, color=COLOR_MILLER, fontweight='bold')

# Cgd
ax1.annotate('', xy=(5, 6), xytext=(1.5, 4.5),
             arrowprops=dict(arrowstyle='-', color=COLOR_RECTA, lw=2.5))
ax1.text(2.5, 5.8, r'$C_{gd}$', fontsize=14, color=COLOR_RECTA, fontweight='bold')
ax1.text(2.5, 5.2, '(Miller!)', fontsize=10, color=COLOR_RECTA, style='italic')

# Cds
ax1.annotate('', xy=(7, 5.5), xytext=(7, 2.5),
             arrowprops=dict(arrowstyle='-', color='gray', lw=1.5))
ax1.text(6, 4, r'$C_{ds}$', fontsize=12, color='gray')

# --- BJT ---
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 8)
ax2.set_aspect('equal')
ax2.axis('off')
ax2.set_title('Modelo AC alta frecuencia - BJT', fontsize=14, fontweight='bold')

# Terminales
for name, xy in [('B', (1, 4)), ('C', (8, 6.5)), ('E', (8, 1.5))]:
    ax2.plot(*xy, 'o', color=COLOR_PRINCIPAL, ms=10, zorder=5)
    ax2.text(xy[0]-0.4 if name=='B' else xy[0]+0.4, xy[1], name,
             fontsize=14, fontweight='bold', color=COLOR_PRINCIPAL, ha='center', va='center')

# gm*Vbe fuente de corriente
ax2.annotate('', xy=(8, 5.5), xytext=(8, 2.5),
             arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2))
ax2.text(9.2, 4, r'$g_m V_{be}$', fontsize=13, color=COLOR_PUNTO, ha='center')

# rpi
ax2.plot([3, 5], [3, 3], color=COLOR_PRINCIPAL, lw=2)
ax2.text(4, 2.5, r'$r_{\pi}$', fontsize=13, color=COLOR_PRINCIPAL, ha='center')

# Cpi
ax2.annotate('', xy=(3.5, 2), xytext=(1.5, 3.5),
             arrowprops=dict(arrowstyle='-', color=COLOR_MILLER, lw=2))
ax2.text(2, 2.2, r'$C_{\pi}$', fontsize=13, color=COLOR_MILLER, fontweight='bold')

# Cmu
ax2.annotate('', xy=(5, 6), xytext=(1.5, 4.5),
             arrowprops=dict(arrowstyle='-', color=COLOR_RECTA, lw=2.5))
ax2.text(2.5, 5.8, r'$C_{\mu}$', fontsize=14, color=COLOR_RECTA, fontweight='bold')
ax2.text(2.5, 5.2, '(Miller!)', fontsize=10, color=COLOR_RECTA, style='italic')

# Ccs
ax2.annotate('', xy=(7, 5.5), xytext=(7, 2.5),
             arrowprops=dict(arrowstyle='-', color='gray', lw=1.5))
ax2.text(6, 4, r'$C_{cs}$', fontsize=12, color='gray')

plt.tight_layout()
plt.show()

---

## 11. Metodologia de resolucion

### Pasos para analizar la respuesta en frecuencia de un amplificador:

**Paso 1:** Identificar la **topologia** del amplificador (CS, CE, CC, CG, CB, etc.)

**Paso 2:** Obtener el **modelo de pequena senal en alta frecuencia** (incluir $C_{gs}$, $C_{gd}$, $C_{ds}$ o sus equivalentes BJT)

**Paso 3:** Calcular la **ganancia en banda media** $A_m = -g_m R_L$ (o la que corresponda)

**Paso 4:** Aplicar el **metodo de constantes de tiempo** para estimar $f_H$:
- Para cada capacidad $C_i$, calcular $R_i$ (resistencia vista con las demas en circuito abierto)
- $\omega_H \approx 1/\sum R_i C_i$

**Paso 5:** Si hay **efecto Miller**, calcular $C_{in} = C_{gs} + C_{gd}(1 + |A_v|)$

**Paso 6:** Calcular el **GBW** y verificar si es adecuado para la aplicacion

**Paso 7:** Representar el **diagrama de Bode** (magnitud y fase)

---

## 12. Ejercicios resueltos

#### Ejercicio resuelto 1: Polo dominante de un CS

**Datos:** $g_m = 4$ mA/V, $R_L = 5$ k$\Omega$, $R_{sig} = 10$ k$\Omega$, $C_{gs} = 2$ pF, $C_{gd} = 0.5$ pF, $C_{ds} = 0.1$ pF

**Paso 1-2:** Ganancia en banda media

$$A_v = -g_m R_L = -4 \times 10^{-3} \times 5 \times 10^3 = -20 \quad (26\;\text{dB})$$

**Paso 3:** Capacidad Miller

$$C_{Miller} = C_{gd}(1 + g_m R_L) = 0.5 \times (1 + 20) = 10.5\;\text{pF}$$

$$C_{in} = C_{gs} + C_{Miller} = 2 + 10.5 = 12.5\;\text{pF}$$

**Paso 4:** Constantes de tiempo

$$\tau_1 = R_{sig} \cdot C_{gs} = 10\text{k} \times 2\text{p} = 20\;\text{ns}$$

$$\tau_2 = [R_{sig}(1 + g_m R_L) + R_L] \cdot C_{gd} = [10\text{k}(21) + 5\text{k}] \times 0.5\text{p} = 107.5\;\text{ns}$$

$$\tau_3 = R_L \cdot C_{ds} = 5\text{k} \times 0.1\text{p} = 0.5\;\text{ns}$$

**Paso 5:** Polo dominante

$$f_H = \frac{1}{2\pi(\tau_1 + \tau_2 + \tau_3)} = \frac{1}{2\pi \times 128\;\text{ns}} = \boxed{1.24\;\text{MHz}}$$

**Paso 6:** GBW

$$GBW = |A_v| \times f_H = 20 \times 1.24\;\text{MHz} = \boxed{24.8\;\text{MHz}}$$

In [ ]:
# Verificacion numerica - Ejercicio 1
gm_e1 = 4e-3
RL_e1 = 5e3
Rsig_e1 = 10e3
Cgs_e1 = 2e-12
Cgd_e1 = 0.5e-12
Cds_e1 = 0.1e-12

Av_e1 = gm_e1 * RL_e1
C_Miller_e1 = Cgd_e1 * (1 + gm_e1 * RL_e1)
Cin_e1 = Cgs_e1 + C_Miller_e1

tau1_e1 = Rsig_e1 * Cgs_e1
tau2_e1 = (Rsig_e1 * (1 + gm_e1 * RL_e1) + RL_e1) * Cgd_e1
tau3_e1 = RL_e1 * Cds_e1
tau_total_e1 = tau1_e1 + tau2_e1 + tau3_e1

fH_e1 = 1 / (2 * np.pi * tau_total_e1)
GBW_e1 = Av_e1 * fH_e1

print(f'Ejercicio 1 - Verificacion:')
print(f'  Av = {Av_e1:.0f} ({20*np.log10(Av_e1):.1f} dB)')
print(f'  C_Miller = {C_Miller_e1*1e12:.1f} pF')
print(f'  Cin = {Cin_e1*1e12:.1f} pF')
print(f'  tau1 = {tau1_e1*1e9:.1f} ns, tau2 = {tau2_e1*1e9:.1f} ns, tau3 = {tau3_e1*1e9:.1f} ns')
print(f'  tau_total = {tau_total_e1*1e9:.1f} ns')
print(f'  fH = {fH_e1/1e6:.2f} MHz')
print(f'  GBW = {GBW_e1/1e6:.1f} MHz')

---

#### Ejercicio resuelto 2: Ancho de banda con realimentacion (GBW)

**Datos:** Amplificador con $A_0 = 10^4$ (80 dB), polo en $f_p = 100$ Hz. Se realimenta para obtener $A_{CL} = 100$.

**Paso 1:** GBW del amplificador

$$GBW = A_0 \times f_p = 10^4 \times 100 = 1\;\text{MHz}$$

**Paso 2:** Ancho de banda con realimentacion

$$BW_{CL} = \frac{GBW}{A_{CL}} = \frac{10^6}{100} = \boxed{10\;\text{kHz}}$$

**Paso 3:** Verificacion

$$A_{CL} \times BW_{CL} = 100 \times 10\;\text{kHz} = 1\;\text{MHz} = GBW \quad \checkmark$$

**Conclusion:** Al reducir la ganancia de $10^4$ a $100$ (factor $100$), el ancho de banda aumenta de $100$ Hz a $10$ kHz (factor $100$). El GBW se conserva.

---

#### Ejercicio resuelto 3: Efecto Miller en amplificador emisor comun (CE)

**Datos:** $g_m = 40$ mA/V, $r_\pi = 2.5$ k$\Omega$, $R_L = 4$ k$\Omega$, $R_{sig} = 5$ k$\Omega$, $C_\pi = 20$ pF, $C_\mu = 1$ pF

**Paso 1:** Ganancia en banda media

$$A_v = -g_m R_L = -40 \times 10^{-3} \times 4 \times 10^3 = -160 \quad (44\;\text{dB})$$

**Paso 2:** Capacidad Miller (equivalente a MOS pero con $C_\mu$ en vez de $C_{gd}$)

$$C_{Miller} = C_\mu(1 + g_m R_L) = 1 \times (1 + 160) = 161\;\text{pF}$$

**Paso 3:** Capacidad total en la entrada (vista desde la base)

La resistencia de base equivalente es $R_B = R_{sig} \parallel r_\pi = \frac{5\text{k} \times 2.5\text{k}}{5\text{k} + 2.5\text{k}} = 1.667\;\text{k}\Omega$

$$C_{in} = C_\pi + C_{Miller} = 20 + 161 = 181\;\text{pF}$$

**Paso 4:** Polo dominante

$$f_H = \frac{1}{2\pi \cdot R_B \cdot C_{in}} = \frac{1}{2\pi \times 1.667\text{k} \times 181\text{p}} = \boxed{527\;\text{kHz}}$$

**Paso 5:** GBW

$$GBW = 160 \times 527\;\text{kHz} = \boxed{84.3\;\text{MHz}}$$

**Nota:** La $C_\mu$ de solo $1$ pF se convierte en $161$ pF por Miller, dominando completamente la respuesta.

In [ ]:
# Verificacion numerica - Ejercicio 3 (CE con BJT)
gm_e3 = 40e-3
rpi_e3 = 2.5e3
RL_e3 = 4e3
Rsig_e3 = 5e3
Cpi_e3 = 20e-12
Cmu_e3 = 1e-12

Av_e3 = gm_e3 * RL_e3
C_Miller_e3 = Cmu_e3 * (1 + gm_e3 * RL_e3)
Cin_e3 = Cpi_e3 + C_Miller_e3
RB_e3 = (Rsig_e3 * rpi_e3) / (Rsig_e3 + rpi_e3)
fH_e3 = 1 / (2 * np.pi * RB_e3 * Cin_e3)
GBW_e3 = Av_e3 * fH_e3

print(f'Ejercicio 3 - Verificacion CE (BJT):')
print(f'  Av = {Av_e3:.0f} ({20*np.log10(Av_e3):.1f} dB)')
print(f'  C_Miller = {C_Miller_e3*1e12:.0f} pF (de solo {Cmu_e3*1e12:.0f} pF!)')
print(f'  Cin = {Cin_e3*1e12:.0f} pF')
print(f'  RB = Rsig || rpi = {RB_e3/1e3:.3f} kOhm')
print(f'  fH = {fH_e3/1e3:.0f} kHz')
print(f'  GBW = {GBW_e3/1e6:.1f} MHz')

---

#### Ejercicio resuelto 4: Diagrama de Bode de un sistema con un cero y dos polos

**Datos:** $A_0 = 50$, polos en $f_{p1} = 1$ kHz y $f_{p2} = 100$ kHz, cero en $f_z = 50$ kHz

$$H(s) = A_0 \cdot \frac{1 + s/(2\pi \cdot 50\text{k})}{(1 + s/(2\pi \cdot 1\text{k}))(1 + s/(2\pi \cdot 100\text{k}))}$$

**Analisis asintotico de magnitud:**

| Rango | Pendiente | Razon |
|-------|-----------|-------|
| $f < 1$ kHz | $0$ dB/dec | Sin efecto de polos ni ceros |
| $1$ k $< f < 50$ kHz | $-20$ dB/dec | Un polo activo |
| $50$ k $< f < 100$ kHz | $0$ dB/dec | Polo + cero se cancelan parcialmente |
| $f > 100$ kHz | $-20$ dB/dec | Dos polos, un cero: neto $-20$ dB/dec |

In [ ]:
# Diagrama de Bode - Sistema con cero y dos polos (Ejercicio 4)
A0_e4 = 50
fp1_e4 = 1e3
fp2_e4 = 100e3
fz_e4 = 50e3

wp1_e4 = 2 * np.pi * fp1_e4
wp2_e4 = 2 * np.pi * fp2_e4
wz_e4 = 2 * np.pi * fz_e4

# H(s) = A0 * (1 + s/wz) / ((1 + s/wp1)*(1 + s/wp2))
num_e4 = [A0_e4/wz_e4, A0_e4]
den_e4 = np.polymul([1/wp1_e4, 1], [1/wp2_e4, 1])
sys_e4 = signal.TransferFunction(num_e4, den_e4)

w_e4 = np.logspace(1, 8, 2000)
w_e4_eval, mag_e4, phase_e4 = signal.bode(sys_e4, w_e4)
f_e4 = w_e4_eval / (2 * np.pi)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Magnitud
ax1.semilogx(f_e4, mag_e4, color=COLOR_PRINCIPAL, lw=2.5, label='Respuesta real')

# Asintotas
mag_asym_e4 = np.zeros_like(f_e4)
for i, f in enumerate(f_e4):
    val = 20 * np.log10(A0_e4)
    if f > fp1_e4:
        val -= 20 * np.log10(f / fp1_e4)
    if f > fz_e4:
        val += 20 * np.log10(f / fz_e4)
    if f > fp2_e4:
        val -= 20 * np.log10(f / fp2_e4)
    mag_asym_e4[i] = val

ax1.semilogx(f_e4, mag_asym_e4, color=COLOR_RECTA, lw=1.5, ls='--', label='Asintotas')
for f_mark, label_mark in [(fp1_e4, '$f_{p1}$'), (fz_e4, '$f_z$'), (fp2_e4, '$f_{p2}$')]:
    ax1.axvline(x=f_mark, color='gray', ls=':', lw=1, alpha=0.5)
    ax1.text(f_mark, -5, label_mark, fontsize=11, ha='center', color='gray')

ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title('Bode - Sistema con 1 cero y 2 polos (Ejercicio 4)', fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, which='both')

# Fase
ax2.semilogx(f_e4, phase_e4, color=COLOR_FASE, lw=2.5)
ax2.axhline(y=-45, color='gray', ls=':', lw=1, alpha=0.5)
ax2.axhline(y=-90, color='gray', ls=':', lw=1, alpha=0.5)
ax2.set_xlabel('Frecuencia (Hz)')
ax2.set_ylabel(r'Fase ($^{\circ}$)')
ax2.set_title('Fase - Sistema con 1 cero y 2 polos', fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---

## 13. Catalogo de tipos de analisis en frecuencia

A continuacion se presentan los 5 tipos mas comunes de problemas de respuesta en frecuencia que aparecen en examenes.

### 13.1 Tipo 1: Diagrama de Bode a partir de $H(s)$

Dada una funcion de transferencia $H(s)$, dibujar el diagrama de Bode.

**Procedimiento:**
1. Identificar $A_0$, polos ($p_j$) y ceros ($z_i$)
2. Calcular la magnitud en banda media: $20\log_{10}(A_0)$ dB
3. En cada polo: $-20$ dB/dec. En cada cero: $+20$ dB/dec
4. Fase: sumar contribuciones de cada polo ($-90^\circ$) y cero ($+90^\circ$)

**Error frecuente:** Olvidar que las asintotas se suman (en dB), no se multiplican.

**Truco:** Dibujar primero la asintota plana, luego anadir las pendientes una a una en orden de frecuencia.

### 13.2 Tipo 2: Calcular $f_H$ con constantes de tiempo (Miller)

Dado un circuito amplificador con parametros del transistor, encontrar el polo dominante.

**Procedimiento:**
1. Calcular $A_v = -g_m R_L$ (ganancia en banda media)
2. Capacidad Miller: $C_{M} = C_{gd}(1 + |A_v|)$
3. $C_{in} = C_{gs} + C_M$
4. $f_H = 1/(2\pi R_{in} C_{in})$ donde $R_{in}$ depende de la topologia

**Truco para el examen:** El termino dominante casi siempre es $C_{gd} \times g_m R_L \times R_{sig}$. Si $g_m R_L \gg 1$, entonces $C_M \gg C_{gs}$ y el Miller domina.

### 13.3 Tipo 3: GBW y ancho de banda con realimentacion

Dado un amplificador con $A_0$ y $f_p$, calcular el BW cuando se aplica realimentacion.

**Procedimiento:**
1. $GBW = A_0 \times f_p$
2. $BW_{CL} = GBW / A_{CL}$
3. Verificar: $A_{CL} \times BW_{CL} = GBW$

**Error frecuente:** Confundir ganancia en lazo abierto ($A_0$) con ganancia en lazo cerrado ($A_{CL}$).

**Nota:** El GBW solo es constante para sistemas de **un polo dominante**. Con dos polos cercanos, el GBW no se conserva exactamente.

### 13.4 Tipo 4: Comparar topologias (CS vs CG vs CC)

Comparar la respuesta en frecuencia de distintas configuraciones del mismo transistor.

| Topologia | $A_v$ | Efecto Miller | $BW$ relativo |
|-----------|-------|---------------|---------------|
| Source Comun (CS) | $-g_m R_L$ (alto) | **Si** (severo) | Bajo |
| Gate Comun (CG) | $+g_m R_L$ (alto) | **No** | Alto |
| Drain Comun (CD/seguidor) | $\approx 1$ | **No** | Muy alto |

**Conclusion clave:** El CG elimina el efecto Miller porque $C_{gd}$ no esta entre entrada y salida. El seguidor de source tiene ganancia $\approx 1$, asi que $C_{gd}$ no se multiplica.

### 13.5 Tipo 5: Determinar si la respuesta es de 1 o 2 polos

A partir de datos experimentales o de la funcion de transferencia, determinar el orden del sistema.

**Indicadores de un solo polo:**
- Pendiente asintotica de $-20$ dB/dec
- Fase tiende a $-90^\circ$
- Caida suave de $-3$ dB en $f_p$

**Indicadores de dos polos:**
- Pendiente asintotica de $-40$ dB/dec (o cambio de $-20$ a $-40$)
- Fase tiende a $-180^\circ$
- Si los polos estan cerca: posible pico de resonancia

**Truco:** Medir la pendiente a alta frecuencia en el Bode. $-20$ dB/dec $\to$ 1 polo. $-40$ dB/dec $\to$ 2 polos. $-60$ dB/dec $\to$ 3 polos.

### Tabla resumen del catalogo

| Tipo | Descripcion | Dato clave | Resultado |
|------|-------------|------------|----------|
| 1 | Bode a partir de $H(s)$ | $A_0$, polos, ceros | Diagrama de magnitud y fase |
| 2 | $f_H$ con Miller | $g_m$, $R_L$, capacidades | Polo dominante |
| 3 | GBW con realimentacion | $A_0$, $f_p$, $A_{CL}$ | $BW_{CL}$ |
| 4 | Comparar topologias | Mismos parametros | BW relativo |
| 5 | Orden del sistema | Bode o $H(s)$ | Numero de polos |

---

## 14. Resumen y formulas clave

| Formula | Uso |
|---------|-----|
| $H(s) = A_m \cdot \dfrac{\prod(1+s/z_i)}{\prod(1+s/p_j)}$ | Funcion de transferencia general |
| $\|H\|_{\text{dB}} = 20\log_{10}\|H\|$ | Conversion a decibelios |
| $H(s) = \dfrac{A_0}{1+s/\omega_p}$, $BW = f_p$ | Sistema de un polo |
| $GBW = A_0 \cdot f_p = A_{CL} \cdot BW_{CL}$ | Producto ganancia-ancho de banda |
| $C_{in} = C_{gs} + C_{gd}(1 + g_m R_L)$ | Efecto Miller (MOS) |
| $C_{in} = C_\pi + C_\mu(1 + g_m R_L)$ | Efecto Miller (BJT) |
| $\omega_H \approx 1/\sum R_i C_i$ | Metodo de constantes de tiempo |
| $f_T = g_m / [2\pi(C_{gs}+C_{gd})]$ | Frecuencia de transicion |
| Polo: $-20$ dB/dec, $-90^\circ$ | Efecto de un polo en Bode |
| Cero: $+20$ dB/dec, $+90^\circ$ | Efecto de un cero en Bode |

**Ideas clave para el examen:**
- El efecto **Miller** multiplica $C_{gd}$ (o $C_\mu$) por la ganancia $\to$ domina la respuesta
- El **GBW** es constante para un polo dominante: ganar ancho de banda cuesta ganancia
- El metodo de **constantes de tiempo** evita calcular toda la funcion de transferencia
- La topologia **gate comun** (o base comun) elimina el efecto Miller
- Dos polos cercanos $\to$ caida abrupta de $-40$ dB/dec y fase hasta $-180^\circ$